In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/annotated/checkpoints/pre_cell_5.pickle

trying: ['pd']
me:  0
trying: ['part']
me:  1
trying: ['load_part']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['load_nation']
me:  4
trying: ['supplier']
me:  3
trying: ['load_supplier']
me:  3
trying: ['nation']
me:  4
trying: ['partsupp']


me:  2
trying: ['region']
me:  5
trying: ['load_partsupp']
me:  2
trying: ['tpch_parent']
me:  0
trying: ['load_region']
me:  5
trying: ['STORAGE_OPTS']
me:  0
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable part
Declaring variable load_part
Declaring variable partsupp
Declaring variable load_partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable load_nation
Declaring variable nation
Declaring variable region
Declaring variable load_region


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# Cell 5 optimized
total = (
    nation[['N_NATIONKEY', 'N_NAME', 'N_REGIONKEY']]
    .merge(
        region[region['R_NAME'] == 'EUROPE'][['R_REGIONKEY']],
        left_on='N_REGIONKEY', right_on='R_REGIONKEY'
    )
    .merge(
        supplier[['S_SUPPKEY', 'S_NAME', 'S_ADDRESS', 'S_NATIONKEY', 'S_PHONE', 'S_ACCTBAL', 'S_COMMENT']],
        left_on='N_NATIONKEY', right_on='S_NATIONKEY'
    )
    .merge(
        partsupp[['PS_PARTKEY', 'PS_SUPPKEY', 'PS_SUPPLYCOST']],
        left_on='S_SUPPKEY', right_on='PS_SUPPKEY'
    )
    .merge(
        part[(part['P_SIZE'] == 15) & (part['P_TYPE'].str.endswith('BRASS'))][['P_PARTKEY', 'P_MFGR']],
        left_on='PS_PARTKEY', right_on='P_PARTKEY'
    )
    .assign(
        MIN_SUPPLYCOST=lambda df: df.groupby('P_PARTKEY')['PS_SUPPLYCOST'].transform('min')
    )
    .query('PS_SUPPLYCOST == MIN_SUPPLYCOST')
    .loc[:, ['S_ACCTBAL', 'S_NAME', 'N_NAME', 'P_PARTKEY', 'P_MFGR', 'S_ADDRESS', 'S_PHONE', 'S_COMMENT']]
    .sort_values(
        by=['S_ACCTBAL', 'N_NAME', 'S_NAME', 'P_PARTKEY'],
        ascending=[False, True, True, True]
    )
)

CPU times: user 556 ms, sys: 232 ms, total: 789 ms
Wall time: 790 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q02/rewritten/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

migration speed (bps): 795613301.1082739
---------------------------
variables to migrate:
DATA_ROOT 80
region 48
STORAGE_OPTS 64
supplier 48
load_region 144
load_nation 144
tpch_parent 54
pd 72
load_supplier 144
part 48
load_part 144
nation 48
partsupp 48
total 48
load_partsupp 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q02/rewritten/o4_mini_high/checkpoints/post_cell_5_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['STORAGE_OPTS', 'DATA_ROOT']
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['STORAGE_OPTS', 'DATA_ROOT']
Active variables []
Intermediate variables ['partsupp']
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables ['STORAGE_OPTS', 'DATA_ROOT']
Active variables []
Intermediate variables ['supplier']
Future variables ['part', 'partsupp']
Modified dataframes
======= Cell 3 =======
Input variables ['STORAGE_OPTS', 'DATA_ROOT']
Active variables []
Intermediate variables ['nation']
Future variables ['part', 'supplier', 'partsupp']
Modified dataframes
======= Cell 4 =======
Input variables ['STORAGE_OPTS', 'DATA_ROOT']
Active variables []
Intermediate variables ['region']
Future variables ['part', 'supplier', 'partsupp', 'nation']
Modified dataframes
======= Cell 5 =======
Input variables ['nation', 'region', 'supplier', 'part', 'partsupp

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q02/opt_cell_exec_info_5_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/annotated/checkpoints/post_cell_5.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
